In [1]:
# 1. import libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix




In [3]:
 
# 2. load data
 
  
import glob
files = [
 "D:\infosys_intern\SentinelNet_Oct_Batch\CIC-IDS\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
 "D:\infosys_intern\SentinelNet_Oct_Batch\CIC-IDS\Friday-WorkingHours-Morning.pcap_ISCX.csv",
 "D:\infosys_intern\SentinelNet_Oct_Batch\CIC-IDS\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
 "D:\infosys_intern\SentinelNet_Oct_Batch\CIC-IDS\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
]
dfs = [pd.read_csv(f, low_memory=False) for f in files]
for d in dfs: d.columns = d.columns.str.strip()
df = pd.concat(dfs, ignore_index=True)
print(f"Data shape: {df.shape}")

Data shape: (875746, 79)


In [4]:
print("Columns:", df.columns.tolist())

Columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE F

In [5]:
# 3. removing non-feature columns
drop_cols = ["Flow ID", "Timestamp", "Source IP", "Destination IP"]
existing = [c for c in drop_cols if c in df.columns]
df.drop(columns=existing, inplace=True)

In [6]:
# 4. data cleaning 
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
print("Cleaned:", df.shape)



Cleaned: (875248, 79)


In [7]:
# 5. encoding target variable
df['Attack'] = df['Label'].astype(str).apply(lambda x: 0 if 'BENIGN' in x.upper() else 1)
df.drop(columns=['Label'], inplace=True)

In [8]:
# 4) ensure all features are numeric
X = df.drop(columns=['Attack'])

X = X.apply(pd.to_numeric, errors='coerce')
y = df['Attack'].astype('int8')

In [9]:
missing_frac = X.isna().mean()
cols_to_drop = missing_frac[missing_frac > 0.4].index.tolist()
if cols_to_drop:
    print("Dropping high-missing columns:", cols_to_drop)
    X.drop(columns=cols_to_drop, inplace=True)
    X = X.apply(pd.to_numeric, errors='coerce')

In [10]:
num_cols = X.select_dtypes(include=[np.number]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

X = X.astype('float32')

In [11]:
# 8) Train/test split 
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class dist:\n", y_train.value_counts())
print("Test class dist:\n", y_test.value_counts())

Train: (700198, 78) Test: (175050, 78)
Train class dist:
 Attack
0    594440
1    105758
Name: count, dtype: int64
Test class dist:
 Attack
0    148611
1     26439
Name: count, dtype: int64


In [12]:
# 9) Scale: fit scaler on original X_train distribution
scaler = StandardScaler()
scaler.fit(X_train)          
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [13]:
# 10) Option A: Use SMOTE only on training set 
sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)
print("After SMOTE (train):", X_train_res.shape, y_train_res.value_counts())



After SMOTE (train): (1188880, 78) Attack
0    594440
1    594440
Name: count, dtype: int64


In [14]:
# 11) PCA: fit on training data  
pca = PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)   
pca.fit(X_train_res)
X_train_pca = pca.transform(X_train_res)
X_test_pca = pca.transform(X_test_scaled)
print("PCA components:", pca.n_components_)

PCA components: 22


In [15]:

import glob, os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
import joblib
from xgboost import XGBClassifier



In [16]:
#model
RSEED = 42
models = {
    "Logistic Regression": LogisticRegression(
        random_state=RSEED, solver='saga', penalty='l2',
        C=0.5, max_iter=5000, class_weight='balanced'
    ),

    "SGD (linear clf)": SGDClassifier(
        loss='log_loss',
        penalty='elasticnet',
        l1_ratio=0.15,
        alpha=1e-4,
        max_iter=2000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RSEED,
        class_weight='balanced'
    ),

    "Gaussian NB": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RSEED,
        criterion='gini',
        max_depth=8,
        min_samples_split=100,
        min_samples_leaf=40,
        ccp_alpha=0.001,
        class_weight='balanced'
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RSEED,
        max_depth=10,
        min_samples_split=100,
        min_samples_leaf=40,
        max_features='sqrt',
        n_jobs=-1,
        class_weight='balanced'
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=10,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=RSEED,
        n_jobs=-1
    ),

   
}


In [17]:
# Train, evaluate, and save models 
results = []
for name, clf in models.items():
    print(f"\nTraining {name} ...")
    clf.fit(X_train_pca, y_train_res)
    probs = clf.predict_proba(X_test_pca)[:,1] if hasattr(clf, "predict_proba") else None
    preds = clf.predict(X_test_pca)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    roc = roc_auc_score(y_test, probs) if probs is not None else np.nan
    pr  = average_precision_score(y_test, probs) if probs is not None else np.nan

    print("Accuracy:", round(acc,4))
    print("Precision:", round(prec,4), "Recall:", round(rec,4), "F1:", round(f1,4))
    if not np.isnan(roc): print("ROC-AUC:", round(roc,4), "PR-AUC:", round(pr,4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, digits=4))

    results.append({
        "model": name, "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1, "roc_auc": roc, "pr_auc": pr
    })

    # Save model artifact
    joblib.dump(clf, f"{name}_model.joblib")
    print("Saved model:", f"{name}_model.joblib")

# Save scaler and pca
joblib.dump(scaler, "scaler.joblib")
joblib.dump(pca, "pca.joblib")
print("Saved scaler.joblib and pca.joblib")

# Summarize results
res_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
print("\nRESULT SUMMARY (sorted by F1):\n", res_df)



Training Logistic Regression ...
Accuracy: 0.9461
Precision: 0.749 Recall: 0.9677 F1: 0.8444
ROC-AUC: 0.9863 PR-AUC: 0.9688
Confusion Matrix:
 [[140039   8572]
 [   855  25584]]
              precision    recall  f1-score   support

           0     0.9939    0.9423    0.9674    148611
           1     0.7490    0.9677    0.8444     26439

    accuracy                         0.9461    175050
   macro avg     0.8715    0.9550    0.9059    175050
weighted avg     0.9569    0.9461    0.9489    175050

Saved model: Logistic Regression_model.joblib

Training SGD (linear clf) ...
Accuracy: 0.9584
Precision: 0.7989 Recall: 0.9681 F1: 0.8754
ROC-AUC: 0.991 PR-AUC: 0.9693
Confusion Matrix:
 [[142167   6444]
 [   843  25596]]
              precision    recall  f1-score   support

           0     0.9941    0.9566    0.9750    148611
           1     0.7989    0.9681    0.8754     26439

    accuracy                         0.9584    175050
   macro avg     0.8965    0.9624    0.9252    175050


c:\Users\HANIF\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [22:47:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.9982
Precision: 0.9954 Recall: 0.9924 F1: 0.9939
ROC-AUC: 0.9999 PR-AUC: 0.9995
Confusion Matrix:
 [[148491    120]
 [   202  26237]]
              precision    recall  f1-score   support

           0     0.9986    0.9992    0.9989    148611
           1     0.9954    0.9924    0.9939     26439

    accuracy                         0.9982    175050
   macro avg     0.9970    0.9958    0.9964    175050
weighted avg     0.9982    0.9982    0.9982    175050

Saved model: XGBoost_model.joblib
Saved scaler.joblib and pca.joblib

RESULT SUMMARY (sorted by F1):
                  model  accuracy  precision    recall        f1   roc_auc  \
4        Random Forest  0.998526   0.997416  0.992814  0.995110  0.999940   
5              XGBoost  0.998161   0.995447  0.992360  0.993901  0.999859   
3        Decision Tree  0.995864   0.985244  0.987405  0.986323  0.997556   
1     SGD (linear clf)  0.958372   0.798876  0.968115  0.875391  0.990976   
0  Logistic Regression  0.946147   0.749

In [18]:
feature_name = X_train.columns.tolist()

import json
with open("selected_features.json", "w") as f:
    json.dump(feature_name, f, indent=4)

print("Saved selected_features.json")


Saved selected_features.json
